# Data Cleaning

This notebook loads the extracted CSVs from `data/02_extraction/`, applies cleaning transformations via `COWCleaner`, and writes clean versions to `data/03_cleaning/`. Cleaning decisions are documented in `src/processing/cleaner.py`; this notebook demonstrates each step individually so intermediate state can be inspected.

## Implementation

Cleaning is implemented by `COWCleaner` in `src/processing/cleaner.py`. Each method is a discrete step that can be called individually for inspection.

| Method | Operation |
|---|---|
| `drop_event_columns()` | Remove spatially redundant and analytically irrelevant event columns |
| `deduplicate_events()` | Keep one row per warning at terminal status (`CAN > COR > EXP > EXT > CON > NEW`) |
| `parse_event_timestamps()` | Parse `issue`/`expire` to UTC datetimes; derive `duration_min` |
| `cap_lead0()` | Cap `lead0` at 99th percentile per phenomena; flag with `lead0_capped` |
| `derive_product_id()` | Add `product_id` join key (`{year}{wfo}{eventid}{phenomena}W1`) |
| `drop_stormreport_columns()` | Remove `type`, `magnitude`, `link` |
| `parse_stormreport_timestamps()` | Parse `valid` to UTC datetime |
| `normalize_source()` | Canonicalize source values; collapse automated stations; impute junk |
| `cap_leadtime()` | Cap `leadtime` at 99th percentile per lsrtype; flag with `leadtime_capped` |
| `impute_null_cities()` | Reverse-geocode null `city` values via Nominatim, with a persisted local cache |
| `impute_null_counties()` | Reverse-geocode null `county` values, sharing the geocode cache |

In [1]:
import logging
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))

from src.cleaning import COWCleaner

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Input: flattened CSVs written by 02_extraction.ipynb
EXTRACTED_DIR = Path("../data/02_extraction")

# Output: cleaned, analysis-ready CSVs for downstream notebooks
PROCESSED_DIR = Path("../data/03_cleaning")

events       = pd.read_csv(EXTRACTED_DIR / "events.csv")
stormreports = pd.read_csv(EXTRACTED_DIR / "stormreports.csv")

print(f"events:       {len(events):,} rows × {len(events.columns)} columns")
print(f"stormreports: {len(stormreports):,} rows × {len(stormreports.columns)} columns")

cleaner = COWCleaner(processed_dir=PROCESSED_DIR)
print(cleaner)

19:02:48  INFO     Loaded geocode cache: 58 entries from ../data/cache/geocode_cache.json


events:       275,276 rows × 32 columns
stormreports: 399,880 rows × 19 columns
COWCleaner(processed=../data/03_cleaning)


## Events

### 1. Drop uninformative columns

Several columns add no analytical value and are dropped before any other work:

- `significance`: constant `'W'` (Warning) across all rows — no variance
- `perimeter`, `areaverify`, `lat0`, `lon0`: spatial geometry fields not used in statistical analysis
- `ar_ugc`, `ar_ugcname`: county/zone codes, not needed at this level of analysis
- `visual_imgurl`, `product_text`, `product_href`, `link`: URL/text blobs with no analytical use
- `stormreports`, `stormreports_all`: embedded JSON lists of storm report IDs; the `stormreports` table is the authoritative source for this relationship
- `fcster`: investigated as a staffing proxy but unusable — format varies within WFOs (badge numbers, last names, initials coexisting), making unique-count comparisons unreliable even within a single office
- `statuses`: identical to `status` after deduplication — a pre-dedup API artifact with no additional information

**Retained for regression (storm-mode covariates):**

- `svr_tornado_possible`: forecaster-flagged boolean on SV warnings indicating a tornado threat was simultaneously present — a direct indicator of prioritization pressure (the forecaster was watching both threats at once)
- `tor_in_svrtorpossible`: boolean indicating a tornado warning was active in the SVR tornado-possible area — a cleaner co-occurrence indicator than a time-window join
- `hailtag`, `windtag`: hail size (inches) and wind speed (knots) thresholds on SV warnings; partial proxies for storm intensity/mode
- `carea`, `parea`: warning polygon area and perimeter; larger warnings tend to be more speculative/precautionary
- `sharedborder`: shared border length with adjacent warnings; proxy for outbreak density

In [2]:
events = cleaner.drop_event_columns(events)
print(f"After drop: {len(events.columns)} columns remaining")
print(events.columns.tolist())

19:02:48  INFO     Dropped 15 event columns; 17 remaining


After drop: 17 columns remaining
['wfo', 'year', 'phenomena', 'eventid', 'sharedborder', 'issue', 'expire', 'svr_tornado_possible', 'hailtag', 'windtag', 'carea', 'parea', 'product_id', 'status', 'verify', 'tor_in_svrtorpossible', 'lead0']


### 2. Deduplicate events

The COW API returns one row per warning issuance status (`NEW`, `CON`, `EXT`, `EXP`, `CAN`, `COR`). For this analysis each warning is one event, so we keep only the terminal status row per `(wfo, year, phenomena, eventid)`. The terminal status is determined by priority: `CAN` > `COR` > `EXP` > `EXT` > `CON` > `NEW`. This preserves the final verified/cancelled state of each warning.

In [3]:
events = cleaner.deduplicate_events(events)
print(f"After deduplication: {len(events):,} rows")
print("\nStatus distribution after deduplication:")
print(events["status"].value_counts().to_string())

19:02:48  INFO     Deduplicated events: 275,276 → 275,163 rows (removed 113)


After deduplication: 275,163 rows

Status distribution after deduplication:
status
EXP    101983
CON     84138
CAN     56179
NEW     31026
EXT      1692
COR       145


### 3. Parse timestamps

`issue` and `expire` are ISO 8601 strings. Parse them to UTC-aware datetimes and derive `duration_min` (warning duration in minutes) as a convenience column.

In [4]:
events = cleaner.parse_event_timestamps(events)
print("issue range:",   events["issue"].min(),  "→", events["issue"].max())
print("expire range:",  events["expire"].min(), "→", events["expire"].max())
print("duration_min — min:", events["duration_min"].min(), "  max:", events["duration_min"].max())
events[["issue", "expire", "duration_min"]].head(3)

19:02:48  INFO     Parsed issue/expire timestamps; derived duration_min


issue range: 2016-01-01 00:36:00+00:00 → 2026-06-17 02:11:00+00:00
expire range: 2016-01-01 01:30:00+00:00 → 2026-06-17 05:15:00+00:00
duration_min — min: -8.0   max: 9224.0


,issue,expire,duration_min
0,2025-02-16 04:26:00+00:00,2025-02-16 09:31:00+00:00,305.0
1,2018-10-02 20:16:00+00:00,2018-10-02 20:29:00+00:00,13.0
2,2025-06-13 23:10:00+00:00,2025-06-13 23:30:00+00:00,20.0


### 4. Inspect `lead0`

`lead0` is the API-supplied lead time in minutes for the first confirming storm report for each verified warning event. It is already numeric in the extracted CSV — no parsing needed. Non-null for all verified events; null for unverified events.

In [5]:
verified = events[events["verify"] == True]
print(f"lead0 non-null: {verified['lead0'].notna().sum():,} of {len(verified):,} verified events")
print()
print(verified["lead0"].describe(percentiles=[.25, .5, .75, .90, .95, .99]).to_string())

lead0 non-null: 129,280 of 129,280 verified events

count    129280.000000
mean         21.035009
std          34.746431
min           0.000000
25%           5.000000
50%          13.000000
75%          25.000000
90%          42.000000
95%          64.000000
99%         165.000000
max        2768.000000


### 5. Cap extreme `lead0` values

FF has a max of ~7,911 minutes (~5.5 days), almost certainly from the API retroactively matching a warning to a storm report days later due to spatial overlap. TO and SV are naturally bounded. Cap per phenomena at the 99th percentile; `lead0_capped` flags affected rows.

In [6]:
events = cleaner.cap_lead0(events)
print(f"Total capped: {events['lead0_capped'].sum():,}")
print("\nPost-cap distribution by phenomena (verified only):")
verified_mask = events["verify"] == True
print(
    events[verified_mask]
    .groupby("phenomena")["lead0"]
    .describe(percentiles=[.25, .5, .75, .95])
    .to_string()
)

19:02:48  INFO     lead0 cap: FF cap=323 min, 182 rows capped
19:02:48  INFO     lead0 cap: SV cap=52 min, 1,014 rows capped
19:02:48  INFO     lead0 cap: TO cap=42 min, 58 rows capped


Total capped: 1,254

Post-cap distribution by phenomena (verified only):
              count       mean        std  min   25%   50%   75%     95%    max
phenomena                                                                      
FF          18202.0  61.801066  62.176469  0.0  18.0  42.0  83.0  188.95  323.0
SV         103830.0  14.291621  12.390363  0.0   4.0  11.0  21.0   40.00   52.0
TO           7248.0  10.432809   9.868623  0.0   2.0   7.0  16.0   31.00   42.0


### 6. Derive join key (`product_id`)

The storm reports table links back to events via a VTEC product ID (e.g. `2020ABQ41SVW1`) stored in the `events` column. The format is `{year}{wfo}{eventid}{phenomena}W1`. Adding this as an explicit column on the events table makes the join unambiguous downstream.

In [7]:
events = cleaner.derive_product_id(events)

# Verify join key coverage
sr_event_ids = (
    stormreports[stormreports["warned"] == True]["events"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)
match_rate = sr_event_ids.isin(events["product_id"]).mean()
print(f"SR → Event match rate via product_id: {match_rate:.2%}")
print("(Remaining misses have significance digit > 1 — co-issued warnings)")
events[["wfo", "year", "phenomena", "eventid", "product_id"]].head(3)

19:02:49  INFO     Derived product_id for 275,163 events


SR → Event match rate via product_id: 99.97%
(Remaining misses have significance digit > 1 — co-issued warnings)


,wfo,year,phenomena,eventid,product_id
0,OHX,2025,FF,12,2025OHX12FFW1
1,PBZ,2018,SV,185,2018PBZ185SVW1
2,LMK,2025,SV,260,2025LMK260SVW1


### 7. Derive seasonal label

Bin each event into a `YYYY-Season` window from its `issue` time. Windows are three months with half-open `[start, next)` boundaries: Spring (Feb 27 - May 26), Summer (May 27 - Aug 26), Fall (Aug 27 - Nov 26), Winter (Nov 27 - Feb 26). Winter wraps the calendar-year boundary; the label year is the window's first day, so Jan 1 - Feb 26 carries the **prior** year's Winter label (e.g. `2026-02-15` -> `2025-Winter`).

The Spring boundary (Feb 27) is the staffing-cut anniversary (`CUT_DATE`), so every `*-Spring` season aligns to the same treatment-relative boundary across all years. The `YYYY-Season` label is distinct from the calendar `year` field: a January row has `year=2026` but `season="2025-Winter"`.

From the label, `derive_season` splits out the columns the models consume, so every time term traces back to the season label rather than the calendar timestamp (the two disagree at the Winter wrap):

| Column | Meaning | Model role |
|---|---|---|
| `season` | the `YYYY-Season` period id | unique period label |
| `season_cat` | season-of-year: Spring / Summer / Fall / Winter | `C(season_cat)`, 4 levels |
| `season_year` | the `YYYY` anchor (window's first-day year) | traceability |
| `study_year` | `season_year - 2015`, a 1-based treatment-relative index | numeric trend term |
| `is_year10` | integer 0/1, 1 where `study_year == 10` | treated indicator |

`study_year` is named to avoid confusion with the calendar year: it is a treatment-relative index (2016 -> 1 ... 2025 -> 10), not a calendar year, and `study_year 10` is the treated period. `is_year10` is derived from `study_year`, never hand-entered.

In [8]:
events = cleaner.derive_season(events, "issue")
print(f"season labels: {events['season'].nunique()} unique")
print(events["season"].value_counts().sort_index().to_string())

19:02:50  INFO     Derived season from 'issue': 43 labels, study_year 0-11


season labels: 43 unique
season
2015-Winter      978
2016-Fall       2569
2016-Spring     6993
2016-Summer    12749
2016-Winter     2001
2017-Fall       2607
2017-Spring     8711
2017-Summer    11927
2017-Winter      551
2018-Fall       3497
2018-Spring     5409
2018-Summer    13786
2018-Winter     1116
2019-Fall       3242
2019-Spring     8235
2019-Summer    15368
2019-Winter     1590
2020-Fall       2233
2020-Spring     6653
2020-Summer    12126
2020-Winter      449
2021-Fall       3304
2021-Spring     5949
2021-Summer    12902
2021-Winter     1453
2022-Fall       2745
2022-Spring     7696
2022-Summer    13042
2022-Winter     1955
2023-Fall       3206
2023-Spring     7682
2023-Summer    17913
2023-Winter     1353
2024-Fall       3193
2024-Spring    10196
2024-Summer    14105
2024-Winter     1195
2025-Fall       3209
2025-Spring    12299
2025-Summer    15116
2025-Winter      773
2026-Spring     8971
2026-Summer     4116


### 8. Clip to the continental US

The study footprint is the continental United States (CONUS). The extracted warnings also carry the non-CONUS offices (Guam, Honolulu, the Alaska offices, and Pago Pago), which have different climatology and reporting practice and are out of scope. `clip_to_conus` drops their rows as an explicit, logged step, by Weather Forecast Office (WFO) code rather than by coordinate, since a warning is an area with no single point. Enforcing the geographic scope once here, on the cleaned table, keeps every downstream consumer CONUS-only without re-deriving the filter in each model fit.

In [9]:
before = len(events)
events = cleaner.clip_to_conus(events)
print(f"kept {len(events):,} of {before:,} events (dropped {before - len(events):,})")
print(f"CONUS offices: {events['wfo'].nunique()}")

19:02:50  INFO     clip_to_conus: dropping 603 of 275,163 rows from 5 non-CONUS offices (AFC=10, AFG=86, AJK=3, GUM=97, HFO=407)
19:02:50  INFO     clip_to_conus: kept 274,560 rows from 116 CONUS offices


kept 274,560 of 275,163 events (dropped 603)
CONUS offices: 116


### 9. Clip to study span

`derive_season` labels every row, including out-of-scope periods: the `2015-Winter` wrap of the earliest January / February data, and 2026 rows collected after the study window closed (collection ran to mid-2026). `clip_to_study_span` drops those as an explicit, logged step, enforcing the ten study seasons (`2016-Spring` through `2025-Winter`) once and globally rather than re-deriving the window in each model fit. The full span is exactly 40 seasons (10 study years x 4 categories), `2016-Spring` through `2025-Winter`.

In [10]:
before = len(events)
events = cleaner.clip_to_study_span(events)
print(f"kept {len(events):,} of {before:,} events (dropped {before - len(events):,})")
print(events.groupby("study_year")["is_year10"].agg(["size", "first"]).to_string())

19:02:50  INFO     clip_to_study_span: dropping 14,023 of 274,560 rows outside 2016-Spring..2025-Winter (2015-Winter=978, 2026-Spring=8,929, 2026-Summer=4,116)
19:02:50  INFO     clip_to_study_span: kept 260,537 rows, study_year 1-10, 31,368 treated rows


kept 260,537 of 274,560 events (dropped 14,023)
             size  first
study_year              
1           24272      0
2           23725      0
3           23721      0
4           28393      0
5           21408      0
6           23551      0
7           25384      0
8           30088      0
9           28627      0
10          31368      1


### 10. Cast outcome to integer

`cast_outcome` converts the boolean `verify` column to integer 0/1, the form the logistic FAR model requires. Doing this in cleaning makes the saved CSV model-ready and avoids a per-fit cast, and it sidesteps a CSV round-trip issue: a boolean column serializes as `True`/`False` text and reloads as object dtype, whereas 0/1 reloads cleanly as integers. The step fails loudly on any NaN or non-boolean value rather than coercing it silently.

In [11]:
events = cleaner.cast_outcome(events, "verify")
print(events["verify"].value_counts().to_string())

19:02:50  INFO     Cast 'verify' to int 0/1: {0: 138012, 1: 122525}


verify
0    138012
1    122525


### 11. Save cleaned events

In [12]:
cleaner.save(events, "events.csv")
print(f"Saved {len(events):,} rows × {len(events.columns)} columns → data/03_cleaning/events.csv")
print(events.columns.tolist())

19:02:53  INFO     Saved 260,537 rows to ../data/03_cleaning/events.csv


Saved 260,537 rows × 24 columns → data/03_cleaning/events.csv
['wfo', 'year', 'phenomena', 'eventid', 'sharedborder', 'issue', 'expire', 'svr_tornado_possible', 'hailtag', 'windtag', 'carea', 'parea', 'product_id', 'status', 'verify', 'tor_in_svrtorpossible', 'lead0', 'duration_min', 'lead0_capped', 'season', 'season_year', 'season_cat', 'study_year', 'is_year10']


## Storm Reports

### 1. Repair malformed rows

A fixed-width field overflow bug in the IEM API corrupts adjacent columns when a long `remark` is serialized, affecting 26 rows across 10 WFOs. Four distinct patterns are repaired in order:

1. **lat0 truncation with clean duplicate** (18 rows) — the corrupt row is dropped; the clean copy is retained
2. **lat0 truncation without clean duplicate** (2 rows — MFL, SGF) — the missing leading digit is inferred from the plausible latitude range and prepended
3. **lon0 sign loss** (5 rows — ILX) — positive CONUS longitude repaired by negating; GUM (Guam/Saipan) with legitimately positive lon0 is excluded
4. **Junk state/county with valid coordinates** (1 row — PIH) — placeholder values nulled so the subsequent Nominatim imputation step resolves them

In [13]:
print(f"Rows before: {len(stormreports):,}")
stormreports = cleaner.repair_malformed_rows(stormreports)
print(f"Rows after:  {len(stormreports):,}")

Rows before: 399,880


19:02:55  WARNING  Dropped 22 corrupt rows with clean duplicates present
19:02:55  WARNING  Repaired lat0: idx=61094 wfo=CLE 1.02 → 41.02
19:02:55  WARNING  Repaired lat0: idx=96268 wfo=EAX 8.92 → 38.92
19:02:55  WARNING  Repaired lat0: idx=96654 wfo=EAX 8.93 → 38.93
19:02:55  WARNING  Repaired lat0: idx=96661 wfo=EAX 8.91 → 38.91
19:02:55  WARNING  Repaired lat0: idx=96760 wfo=EAX 8.91 → 38.91
19:02:55  WARNING  Repaired lat0: idx=97268 wfo=EAX 8.91 → 38.91
19:02:55  WARNING  Repaired lat0: idx=152008 wfo=GSP 5.87 → 35.87
19:02:55  WARNING  Repaired lat0: idx=262589 wfo=MFL 6.2 → 26.2
19:02:55  WARNING  Repaired lat0: idx=358929 wfo=SGF 7.84 → 37.84
19:02:55  WARNING  Repaired lon0 sign for 6 rows
19:02:55  WARNING  Nulled junk state/county for 14 rows


Rows after:  399,858


### 2. Drop uninformative columns

- `type`: single-character raw code that duplicates `lsrtype`; the latter is more readable
- `magnitude`: 67% null; magnitude values are not used in any planned analysis
- `link`: URL blob with no analytical use

`lon0` and `lat0` are retained for potential spatial EDA. `events` is retained as the trace link from each storm report back to its matched warning(s) — it is a comma-separated list of VTEC product IDs (e.g. `2020ABQ41SVW1`) and is null only for unwarned reports.

In [14]:
stormreports = cleaner.drop_stormreport_columns(stormreports)
print(f"After drop: {len(stormreports.columns)} columns remaining")
print(stormreports.columns.tolist())

19:02:55  INFO     Dropped 3 stormreport columns; 16 remaining


After drop: 16 columns remaining
['wfo', 'year', 'valid', 'city', 'county', 'state', 'source', 'remark', 'typetext', 'lon0', 'lat0', 'events', 'tdq', 'warned', 'leadtime', 'lsrtype']


### 3. Parse timestamp

`valid` is an ISO 8601 string. Parse to UTC-aware datetime.

In [15]:
stormreports = cleaner.parse_stormreport_timestamps(stormreports)
print("valid range:", stormreports["valid"].min(), "→", stormreports["valid"].max())

19:02:55  INFO     Parsed stormreports.valid timestamps


valid range: 2016-01-03 01:35:00+00:00 → 2026-06-17 01:26:00+00:00


### 4. Normalize `source`

Four issues addressed in order:

1. **Case and whitespace** — strip and uppercase collapses 132 → 91 unique values
2. **Truncation artifacts** — values where the leading character(s) were clipped (e.g. `MERGENCY MNGR`, `UBLIC`, `ROADCAST MEDIA`) mapped back to their canonical form
3. **Abbreviation variants** — `COUNTY EMERGY MG` / `COUNTY EMERGY MGMT` → `COUNTY OFFICIAL`; `NEWSPAPER` → `PRINT MEDIA`; `SOCIAL MEDIA` retained as distinct from `BROADCAST MEDIA` and `PRINT MEDIA`; automated station codes (`WEATHERFLOW`, `WXFLOW`, `MESOWEST`, etc.) → `AUTOMATED STATION`
4. **Null and junk values** (`X`, `NONE`, `UNKNOWN`, `FIRE`) — imputed as `"not_provided"`

In [16]:
stormreports = cleaner.normalize_source(stormreports)
print(f"Null source: {stormreports['source'].isnull().sum()}")
print()
print(stormreports["source"].value_counts().to_string())

19:02:56  INFO     Normalized source: 185 → 91 unique values


Null source: 0

source
PUBLIC                    98463
EMERGENCY MNGR            56189
911 CALL CENTER           48440
TRAINED SPOTTER           44496
LAW ENFORCEMENT           30537
BROADCAST MEDIA           18660
MESONET                   18032
NWS STORM SURVEY          11216
DEPT OF HIGHWAYS          10023
AMATEUR RADIO              9798
FIRE DEPT/RESCUE           9376
NWS EMPLOYEE               7419
ASOS                       7273
UTILITY COMPANY            5173
STORM CHASER               4754
SOCIAL MEDIA               4716
COUNTY OFFICIAL            3353
AWOS                       3034
CO-OP OBSERVER             2359
OTHER FEDERAL              1291
COCORAHS                   1218
PARK/FOREST SRVC            872
PRINT MEDIA                 866
OFFICIAL NWS OBS            846
TWITTER                     329
FACEBOOK                    277
POST OFFICE                 145
AUTOMATED STATION           143
not_provided                 95
NWS OFFICE                   46
SHERIFF OFFICE   

### 5. Inspect `leadtime`

`leadtime` is the API-supplied lead time in minutes between warning issuance and each storm report. It is already numeric in the extracted CSV — no parsing needed. Non-null for all warned reports; null for unwarned reports.

In [17]:
warned_sr = stormreports[stormreports["warned"] == True]
print(f"leadtime non-null: {warned_sr['leadtime'].notna().sum():,} of {len(warned_sr):,} warned rows")
print()
print(warned_sr["leadtime"].describe(percentiles=[.25, .5, .75, .90, .95, .99]).to_string())

leadtime non-null: 306,042 of 306,042 warned rows

count    306042.000000
mean         35.118454
std          69.409585
min           0.000000
25%          12.000000
50%          23.000000
75%          37.000000
90%          58.000000
95%         108.000000
99%         277.000000
max        7996.000000


### 6. Cap extreme `leadtime` values

`leadtime` is API-supplied, computed server-side as the difference in minutes between warning issuance and storm report time. FF warnings can spatially overlap storm reports from days later, producing physically implausible values (max ~7,996 min). TO and SV are naturally bounded. Cap per lsrtype at the 99th percentile; `leadtime_capped` flags affected rows.

In [18]:
stormreports = cleaner.cap_leadtime(stormreports)
print(f"Total capped: {stormreports['leadtime_capped'].sum():,}")
print("\nPost-cap distribution by lsrtype (warned only):")
warned_mask = stormreports["warned"] == True
print(
    stormreports[warned_mask]
    .groupby("lsrtype")["leadtime"]
    .describe(percentiles=[.25, .5, .75, .95])
    .to_string()
)

19:02:56  INFO     leadtime cap: FF cap=580 min, 443 rows capped
19:02:56  INFO     leadtime cap: SV cap=61 min, 2,439 rows capped
19:02:56  INFO     leadtime cap: TO cap=49 min, 117 rows capped


Total capped: 2,999

Post-cap distribution by lsrtype (warned only):
            count        mean         std  min   25%   50%    75%     95%    max
lsrtype                                                                         
FF        44310.0  105.080050  103.784133  0.0  35.0  74.0  139.0  314.55  580.0
SV       249638.0   22.815092   14.479189  0.0  11.0  21.0   33.0   49.00   61.0
TO        12094.0   15.335952   11.737115  0.0   6.0  13.0   23.0   38.00   49.0


### 7. Flag TDQ reports

`tdq` ("Too Difficult to Qualify") marks reports that NWS could not verify as a true hazardous event. These are retained in the dataset but flagged — downstream analyses that compute POD should consider excluding them to avoid inflating verified event counts.

In [19]:
tdq_count = stormreports["tdq"].sum()
print(f"TDQ reports: {tdq_count:,} of {len(stormreports):,} ({tdq_count/len(stormreports):.1%})")
print()
print("TDQ by lsrtype:")
print(stormreports.groupby("lsrtype")["tdq"].sum().to_string())

TDQ reports: 7,801 of 399,858 (2.0%)

TDQ by lsrtype:
lsrtype
FF      93
SV    7708
TO       0


### 8. Impute null `city` values

A small number of rows have no city. For each, reverse-geocode `lon0`/`lat0` via the Nominatim API and use the most specific named place available (city → town → village → hamlet), falling back to `"not_provided"` if the location is too rural to resolve.

Resolved coordinates are cached to `data/cache/geocode_cache.json`, keyed by rounded `(lat, lon)`. The cache is shared with county imputation and persists across runs, so reruns skip the API call (and its one-second rate limit) for any coordinate already seen.

In [20]:
print(f"Null city before: {stormreports['city'].isnull().sum()}")
stormreports = cleaner.impute_null_cities(stormreports)
print(f"Null city after:  {stormreports['city'].isnull().sum()}")

19:02:56  INFO     Imputing 5 null city values (cache: 58 entries) ...
19:02:56  INFO       city imputed: idx=121482 (43.1, -96.19) → 'Sioux Center'
19:02:56  INFO       city imputed: idx=121505 (44.12, -95.74) → 'not_provided'
19:02:56  INFO       city imputed: idx=147366 (34.29, -82.25) → 'Hodges'
19:02:56  INFO       city imputed: idx=147835 (34.35, -82.06) → 'Waterloo'
19:02:56  INFO       city imputed: idx=147846 (34.11, -82.25) → 'Promised Land'
19:02:56  INFO     city imputation: 0 new cache entries
19:02:56  INFO     Saved geocode cache: 58 entries


Null city before: 5
Null city after:  0


### 9. Impute null `county` values

18 rows (all AJK 2020, southeast Alaska) have no county. Alaska uses boroughs rather than counties, and some areas are part of the Unorganized Borough — Nominatim may return `county`, `city` (for unified city-boroughs like Juneau), or nothing. Use the same reverse-geocode approach, falling back through `county` → `city` → `state_district` → `"not_provided"`.

This reuses the geocode cache populated during city imputation, so the same coordinates resolve with no further API calls.

In [21]:
print(f"Null county before: {stormreports['county'].isnull().sum()}")
stormreports = cleaner.impute_null_counties(stormreports)
print(f"Null county after:  {stormreports['county'].isnull().sum()}")

Null county before: 54


19:02:56  INFO     Imputing 54 null county values (cache: 58 entries) ...
19:02:56  INFO       county imputed: idx=5904 (61.81, -147.5) → 'Matanuska-Susitna Borough'
19:02:56  INFO       county imputed: idx=5905 (61.08, -146.3) → 'Unorganized Borough'
19:02:56  INFO       county imputed: idx=5906 (61.16, -149.9) → 'Anchorage'
19:02:56  INFO       county imputed: idx=5925 (63.77, -145.95) → 'Unorganized Borough'
19:02:56  INFO       county imputed: idx=5926 (63.44, -150.27) → 'Denali Borough'
19:02:56  INFO       county imputed: idx=5953 (55.21, -132.82) → 'Unorganized Borough'
19:02:56  INFO       county imputed: idx=5954 (57.06, -135.36) → 'Sitka'
19:02:56  INFO       county imputed: idx=5955 (57.05, -135.23) → 'Sitka'
19:02:56  INFO       county imputed: idx=5956 (57.12, -135.39) → 'Sitka'
19:02:56  INFO       county imputed: idx=5957 (58.28, -134.37) → 'Juneau'
19:02:56  INFO       county imputed: idx=5958 (58.3, -134.41) → 'Juneau'
19:02:56  INFO       county imputed: idx=5959 (59.

Null county after:  0


### 10. Derive seasonal label

Bin each storm report into a `YYYY-Season` window from its `valid` time, using the same scheme as the events table (see Events step 7). Storm reports bin on their own `valid` timestamp rather than `issue`, which the table does not carry. The same `season`, `season_cat`, `season_year`, `study_year`, and `is_year10` columns are derived.

In [22]:
stormreports = cleaner.derive_season(stormreports, "valid")
print(f"season labels: {stormreports['season'].nunique()} unique")
print(stormreports["season"].value_counts().sort_index().to_string())

19:02:57  INFO     Derived season from 'valid': 43 labels, study_year 0-11


season labels: 43 unique
season
2015-Winter     1740
2016-Fall       3080
2016-Spring     9123
2016-Summer    18820
2016-Winter     3122
2017-Fall       3126
2017-Spring    14023
2017-Summer    17265
2017-Winter      758
2018-Fall       4935
2018-Spring     7526
2018-Summer    18603
2018-Winter     1928
2019-Fall       4279
2019-Spring    10603
2019-Summer    21950
2019-Winter     3318
2020-Fall       3953
2020-Spring    11439
2020-Summer    19081
2020-Winter      679
2021-Fall       4687
2021-Spring     8044
2021-Summer    18197
2021-Winter     2497
2022-Fall       2800
2022-Spring    10898
2022-Summer    19297
2022-Winter     2627
2023-Fall       3945
2023-Spring    10598
2023-Summer    27256
2023-Winter     2461
2024-Fall       4160
2024-Spring    15390
2024-Summer    20701
2024-Winter     2504
2025-Fall       3211
2025-Spring    18290
2025-Summer    20631
2025-Winter     1729
2026-Spring    13706
2026-Summer     6878


### 11. Clip to the continental US

Apply the same CONUS restriction as the events table (see Events step 8): drop rows from the non-CONUS offices (Guam, Honolulu, the Alaska offices, and Pago Pago) by Weather Forecast Office (WFO) code, so both cleaned tables share the same continental US footprint.

In [23]:
before = len(stormreports)
stormreports = cleaner.clip_to_conus(stormreports)
print(f"kept {len(stormreports):,} of {before:,} storm reports (dropped {before - len(stormreports):,})")
print(f"CONUS offices: {stormreports['wfo'].nunique()}")

19:02:58  INFO     clip_to_conus: dropping 690 of 399,858 rows from 5 non-CONUS offices (AFC=21, AFG=28, AJK=93, GUM=23, HFO=525)
19:02:58  INFO     clip_to_conus: kept 399,168 rows from 116 CONUS offices


kept 399,168 of 399,858 storm reports (dropped 690)
CONUS offices: 116


### 12. Clip to study span

Apply the same study-span restriction as the events table (see Events step 9): drop the `2015-Winter` wrap and post-window 2026 rows so the storm reports cover exactly `2016-Spring` through `2025-Winter`.

In [24]:
before = len(stormreports)
stormreports = cleaner.clip_to_study_span(stormreports)
print(f"kept {len(stormreports):,} of {before:,} storm reports (dropped {before - len(stormreports):,})")
print(stormreports.groupby("study_year")["is_year10"].agg(["size", "first"]).to_string())

19:02:58  INFO     clip_to_study_span: dropping 22,205 of 399,168 rows outside 2016-Spring..2025-Winter (2015-Winter=1,739, 2026-Spring=13,588, 2026-Summer=6,878)
19:02:58  INFO     clip_to_study_span: kept 376,963 rows, study_year 1-10, 43,820 treated rows


kept 376,963 of 399,168 storm reports (dropped 22,205)
             size  first
study_year              
1           34104      0
2           35125      0
3           32923      0
4           40121      0
5           35098      0
6           33350      0
7           35566      0
8           44201      0
9           42655      0
10          43820      1


### 13. Cast outcome to integer

Convert the boolean `warned` column (the POD outcome) to integer 0/1, the same treatment applied to `verify` on events (see Events step 10).

In [25]:
stormreports = cleaner.cast_outcome(stormreports, "warned")
print(stormreports["warned"].value_counts().to_string())

19:02:58  INFO     Cast 'warned' to int 0/1: {1: 287832, 0: 89131}


warned
1    287832
0     89131


### 14. Save cleaned storm reports

In [26]:
cleaner.save(stormreports, "stormreports.csv")
print(f"Saved {len(stormreports):,} rows × {len(stormreports.columns)} columns → data/03_cleaning/stormreports.csv")
print(stormreports.columns.tolist())

19:03:01  INFO     Saved 376,963 rows to ../data/03_cleaning/stormreports.csv


Saved 376,963 rows × 22 columns → data/03_cleaning/stormreports.csv
['wfo', 'year', 'valid', 'city', 'county', 'state', 'source', 'remark', 'typetext', 'lon0', 'lat0', 'events', 'tdq', 'warned', 'leadtime', 'lsrtype', 'leadtime_capped', 'season', 'season_year', 'season_cat', 'study_year', 'is_year10']


## Diagnostics

In [27]:
events.head()

,wfo,year,phenomena,eventid,sharedborder,issue,expire,svr_tornado_possible,hailtag,windtag,...,verify,tor_in_svrtorpossible,lead0,duration_min,lead0_capped,season,season_year,season_cat,study_year,is_year10
0,OHX,2025,FF,12,30346.760714,2025-02-16 04:26:00+00:00,2025-02-16 09:31:00+00:00,False,NaN,NaN,...,0,False,NaN,305.0,False,2024-Winter,2024,Winter,9,0
1,PBZ,2018,SV,185,21349.537250,2018-10-02 20:16:00+00:00,2018-10-02 20:29:00+00:00,True,0.75,60.0,...,0,False,NaN,13.0,False,2018-Fall,2018,Fall,3,0
2,LMK,2025,SV,260,26670.070004,2025-06-13 23:10:00+00:00,2025-06-13 23:30:00+00:00,True,0.75,60.0,...,0,False,NaN,20.0,False,2025-Summer,2025,Summer,10,1
3,CYS,2024,FF,14,56674.092307,2024-07-28 02:07:00+00:00,2024-07-28 04:01:00+00:00,False,NaN,NaN,...,1,False,3.0,114.0,False,2024-Summer,2024,Summer,9,0
4,PBZ,2018,SV,173,0.000000,2018-09-06 20:32:00+00:00,2018-09-06 20:49:00+00:00,False,1.00,60.0,...,1,False,3.0,17.0,False,2018-Fall,2018,Fall,3,0


In [28]:
events.info()

<class 'pandas.DataFrame'>
Index: 260537 entries, 0 to 275162
Data columns (total 24 columns):
 #   Column                 Non-Null Count   Dtype              
---  ------                 --------------   -----              
 0   wfo                    260537 non-null  str                
 1   year                   260537 non-null  int64              
 2   phenomena              260537 non-null  str                
 3   eventid                260537 non-null  int64              
 4   sharedborder           260526 non-null  float64            
 5   issue                  260537 non-null  datetime64[us, UTC]
 6   expire                 260537 non-null  datetime64[us, UTC]
 7   svr_tornado_possible   260537 non-null  bool               
 8   hailtag                220630 non-null  float64            
 9   windtag                195014 non-null  float64            
 10  carea                  260537 non-null  float64            
 11  parea                  260537 non-null  float64        

In [29]:
stormreports.head()

,wfo,year,valid,city,county,state,source,remark,typetext,lon0,...,tdq,warned,leadtime,lsrtype,leadtime_capped,season,season_year,season_cat,study_year,is_year10
1,ABQ,2016,2016-04-15 23:25:00+00:00,3 E SENECA,UNION,NM,TRAINED SPOTTER,NaN,HAIL,-103.07,...,False,1,31.0,SV,False,2016-Spring,2016,Spring,1,0
2,ABQ,2016,2016-04-22 21:55:00+00:00,3 WSW BELEN,VALENCIA,NM,AWOS,KE80 AWOS.,TSTM WND GST,-106.83,...,False,0,NaN,SV,False,2016-Spring,2016,Spring,1,0
3,ABQ,2016,2016-04-23 01:00:00+00:00,2 ENE CLAYTON,UNION,NM,ASOS,KCAO ASOS.,TSTM WND GST,-103.14,...,False,0,NaN,SV,False,2016-Spring,2016,Spring,1,0
4,ABQ,2016,2016-04-24 00:19:00+00:00,3 NNE LA CIENEGA,SANTA FE,NM,ASOS,KSAF ASOS.,TSTM WND GST,-106.11,...,False,0,NaN,SV,False,2016-Spring,2016,Spring,1,0
5,ABQ,2016,2016-04-24 04:05:00+00:00,4 WSW MILLS,HARDING,NM,PARK/FOREST SRVC,MILLS CANYON RAWS.,TSTM WND GST,-104.32,...,False,0,NaN,SV,False,2016-Spring,2016,Spring,1,0


In [30]:
stormreports.info()

<class 'pandas.DataFrame'>
Index: 376963 entries, 1 to 399853
Data columns (total 22 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   wfo              376963 non-null  str                
 1   year             376963 non-null  int64              
 2   valid            376963 non-null  datetime64[us, UTC]
 3   city             376963 non-null  str                
 4   county           376963 non-null  str                
 5   state            376952 non-null  str                
 6   source           376963 non-null  str                
 7   remark           328873 non-null  str                
 8   typetext         376963 non-null  str                
 9   lon0             376963 non-null  float64            
 10  lat0             376963 non-null  float64            
 11  events           300717 non-null  str                
 12  tdq              376963 non-null  bool               
 13  warned         